In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import datetime as dt
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, balanced_accuracy_score, confusion_matrix, precision_recall_curve, roc_auc_score, accuracy_score, f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.impute import KNNImputer
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE


## 中興Paper

In [ ]:
secom = pd.read_csv(r'C:\Users\No\Documents\GitHub\psychic-spoon\DataSet\secom.csv', sep='\t')

# 準備 X, y
X = secom.drop(columns=['Time', 'Pass/Fail']).apply(pd.to_numeric, errors='coerce')
y = secom['Pass/Fail'].replace({-1:1, 1:0}).astype('int')

# (1) 移除缺失 > 1000 的特徵
na_counts = X.isna().sum()
X = X.loc[:, na_counts<=1000].copy()

# (2) 切分
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# (3) KNN 填補（k=10）
imputer = KNNImputer(n_neighbors=10)
X_tr_impute = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_te_impute = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns, index=X_test.index)

# (4) 隨機森林重要度：取前 140 特徵
rfc = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight='balanced')
rfc.fit(X_tr_impute, y_train)
X_impute_series = pd.Series(rfc.feature_importances_, index=X_tr_impute.columns).sort_values(ascending=False)
top_k = X_impute_series.index[:140].tolist()

X_tr_top = X_tr_impute[top_k].copy()
X_te_top = X_te_impute[top_k].copy()

# 5) SMOTE（sampling_strategy=0.5）
smote = SMOTE(sampling_strategy=0.5, k_neighbors=5, random_state=42)
X_tr_balanced, y_tr_balanced = smote.fit_resample(X_tr_top, y_train)

# (6) 決策樹（gini、max_depth=6）
dtc = DecisionTreeClassifier(criterion='gini', max_depth=6, random_state=42)
dtc.fit(X_tr_balanced, y_tr_balanced)

y_pred = dtc.predict(X_te_top)
y_prob = dtc.predict_proba(X_te_top)[:, 1] # 以「1」為正類計算 ROC-AUC
auc_roc = roc_auc_score(y_test, y_prob)

print("\n=== Decision Tree (criterion=gini, max_depth=6) ===")

print(classification_report(y_test, y_pred, digits=4))
print("ROC-AUC:", round(auc_roc, 6))

cm = confusion_matrix(y_test, y_pred, labels=[0,1])  # [[TN, FP],[FN, TP]]
TP, FN, FP, TN = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
FPR = FP / (FP + TN) if (FP + TN) > 0 else np.nan
print("Confusion Matrix:\n", cm)
print("FPR = ", round(FPR*100, 2), "%")

sensitivy = TP/(TP+FN) if (TP+FN)>0 else np.nan
specificity = TN/(TN+FP) if (TN+FP)>0 else np.nan
GM = (sensitivy*specificity)**(1/2)
print("sensitivy = ", round(sensitivy*100, 2), "%")
print("specificity = ", round(specificity*100, 2), "%")
print("GM = ", round(GM*100, 2), "%")


---
---

In [3]:
secom = pd.read_csv(r'C:\Users\No\Documents\GitHub\psychic-spoon\DataSet\secom.csv', sep='\t')

secom = secom.replace(['NaN'], np.nan)
numeric_cols = secom.drop(columns=['Time', 'Pass/Fail']).select_dtypes(include=[np.number]).columns.tolist()
knn = KNNImputer(n_neighbors=5, weights='distance')
secom[numeric_cols] = knn.fit_transform(secom[numeric_cols])

secom['Pass/Fail'] = secom['Pass/Fail'].replace({-1:1, 1:0})
# X = 特徵, y = 目標
X = secom.drop(columns=['Time', 'Pass/Fail'])
y = secom['Pass/Fail']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [4]:
# === 1) Linear Regression ===
lir_ = LinearRegression()
model1 = lir_.fit(X_train, y_train)
y_pred1 = model1.predict(X_test)
y_pred1_cls = (y_pred1 >= 0.5).astype(int)
print("\n=== Linear Regression ===")
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred1_cls, digits=4))

cm = confusion_matrix(y_test, y_pred1_cls, labels=[0,1])  # [[TN, FP],[FN, TP]]
TP, FN, FP, TN = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
FPR = FP / (FP + TN) if (FP + TN) > 0 else np.nan
print("Confusion Matrix:\n", cm)
print("FPR = ", round(FPR*100, 2), "%")

sensitivy = TP/(TP+FN) if (TP+FN)>0 else np.nan
specificity = TN/(TN+FP) if (TN+FP)>0 else np.nan
GM = (sensitivy*specificity)**(1/2)
print("sensitivy = ", round(sensitivy*100, 2), "%")
print("specificity = ", round(specificity*100, 2), "%")
print("GM = ", round(GM*100, 2), "%")



=== Linear Regression ===

=== Classification Report ===
              precision    recall  f1-score   support

           0     0.3333    0.1935    0.2449        31
           1     0.9448    0.9727    0.9586       440

    accuracy                         0.9214       471
   macro avg     0.6391    0.5831    0.6017       471
weighted avg     0.9046    0.9214    0.9116       471

Confusion Matrix:
 [[  6  25]
 [ 12 428]]
FPR =  2.73 %
sensitivy =  19.35 %
specificity =  97.27 %
GM =  43.39 %


In [5]:
# === 5) XGBoost ===
xgb = XGBClassifier(random_state=42, n_jobs=-1)
model5 = xgb.fit(X_train, y_train)
y_pred5 = model5.predict(X_test)
print("\n=== XGBoost ===")
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred5, digits=4))

cm = confusion_matrix(y_test, y_pred5, labels=[0,1])  # [[TP, FN],[FP, TN]]
TP, FN, FP, TN = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
FPR = FP / (FP + TN) if (FP + TN) > 0 else np.nan
print("Confusion Matrix:\n", cm)
print("FPR = ", round(FPR*100, 2), "%")

sensitivy = TP/(TP+FN) if (TP+FN)>0 else np.nan
specificity = TN/(TN+FP) if (TN+FP)>0 else np.nan
GM = (sensitivy*specificity)**(1/2)
print("sensitivy = ", round(sensitivy*100, 2), "%")
print("specificity = ", round(specificity*100, 2), "%")
print("GM = ", round(GM*100, 2), "%")



=== XGBoost ===

=== Classification Report ===
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000        31
           1     0.9342    1.0000    0.9660       440

    accuracy                         0.9342       471
   macro avg     0.4671    0.5000    0.4830       471
weighted avg     0.8727    0.9342    0.9024       471

Confusion Matrix:
 [[  0  31]
 [  0 440]]
FPR =  0.0 %
sensitivy =  0.0 %
specificity =  100.0 %
GM =  0.0 %


c:\Users\No\anaconda3\envs\Laptop_20250527\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\No\anaconda3\envs\Laptop_20250527\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\No\anaconda3\envs\Laptop_20250527\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is",